<a href="https://colab.research.google.com/github/Mozhan/ETL_pipeline_examples/blob/main/Weather_ETL_pipeline_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import requests
import pandas as pd
from datetime import datetime, UTC

# -------------------------------------------------------------
# 1. EXTRACT LAYER: Fetch data from a live Public API
# -------------------------------------------------------------
def extract_weather_data(city: str) -> dict:
    """Fetches live weather data from the Open-Meteo API (No API key required)."""
    # Using the true API endpoint with coordinates for Amsterdam.
    # For dynamic city input, a geocoding service would be needed to get lat/long.
    latitude = 52.37
    longitude = 4.89
    url = f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true"

    response = requests.get(url)

    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"API request failed with status code: {response.status_code}. Response: {response.text}")

# -------------------------------------------------------------
# 2. TRANSFORM LAYER: Clean, filter, and structure the data
# -------------------------------------------------------------
def transform_data(raw_json: dict, city: str) -> pd.DataFrame:
    """Parses raw JSON data into a clean, flat analytical DataFrame."""
    current_weather = raw_json.get("current_weather", {})

    # Extract only the metrics an analyst would care about
    extracted_metrics = {
        "city": city,
        "temperature_celsius": current_weather.get("temperature"),
        "windspeed_kmh": current_weather.get("windspeed"),
        "weather_code": current_weather.get("weathercode"),
        "extracted_at": datetime.now(UTC).strftime("%Y-%m-%d %H:%M:%S")
    }

    # Convert dictionary to a structured Pandas DataFrame
    df = pd.DataFrame([extracted_metrics])

    # Convert temperature to numeric and check for non-numeric values
    df['temperature_celsius'] = pd.to_numeric(df['temperature_celsius'], errors='coerce')
    if df['temperature_celsius'].isnull().any():
        raise ValueError("Data anomaly detected: Temperature value is not numeric!")

    # Data Quality Check: Ensure temperature values are reasonable (already present, but good to keep)
    if df["temperature_celsius"].isnull().values.any():
        raise ValueError("Data anomaly detected: Temperature value is missing!")

    return df

# -------------------------------------------------------------
# 3. LOAD LAYER (Conceptual): Write to local disk or push to AWS S3
# -------------------------------------------------------------
def load_data(df: pd.DataFrame, output_filename: str):
    """Saves the data locally as a CSV. In production, this pushes to AWS S3."""
    # Append data if the file already exists, otherwise create new
    if not os.path.isfile(output_filename):
        df.to_csv(output_filename, index=False)
    else:
        df.to_csv(output_filename, mode='a', header=False, index=False)

    print(f"Successfully loaded data into {output_filename}")

# -------------------------------------------------------------
# MAIN EXECUTION PIPELINE
# -------------------------------------------------------------
if __name__ == "__main__":
    CITY_NAME = "Amsterdam"
    OUTPUT_FILE = "daily_weather_log.csv"

    print(f"Kicking off ETL pipeline for {CITY_NAME}...")

    try:
        # Step 1: Extract
        raw_data = extract_weather_data(CITY_NAME)

        # Step 2: Transform
        cleaned_df = transform_data(raw_data, CITY_NAME)

        # Step 3: Load
        load_data(cleaned_df, OUTPUT_FILE)

        print("Pipeline execution completed successfully!")
        print(cleaned_df)

    except Exception as e:
        print(f"Pipeline Failed: {e}")